# Solve the Audio Classification

In [ ]:
import matplotlib.pyplot as plt
import numpy as np 
import os
import pandas as pd 
import seaborn as sns
import timm
import torch
import torch.nn.functional as F

from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
target_cols = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
colors = [ "red", "orange", "yellow", "green", "blue", "purple",]

eegs_train_dir = "/kaggle/input/competitions/hms-harmful-brain-activity-classification/train_eegs"
spectogram_train_dir = "/kaggle/input/competitions/hms-harmful-brain-activity-classification/train_spectrograms"
eegs_test_dir = "/kaggle/input/competitions/hms-harmful-brain-activity-classification/test_eegs"
spectogram_test_dir = "/kaggle/input/competitions/hms-harmful-brain-activity-classification/test_spectrograms"

device = "cuda" if torch.cuda.is_available() else "cpu"

train_len = None
val_len = None

num_classes = len(target_cols)
batch_size = 32
n_epochs = 5 

In [ ]:
def metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    uar = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return [accuracy, uar, macro_f1]

In [ ]:
def kl_loss(logits, targets):
    log_probs = F.log_softmax(logits, dim=1)

    return F.kl_div(
        log_probs,
        targets,
        reduction="batchmean"
    )

## Data

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/hms-harmful-brain-activity-classification/train.csv")
# df = df[["spectrogram_id", "eeg_id", "patient_id", "expert_consensus", "seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]]
# train, val = train_test_split(df, stratify=df['expert_consensus'], groups=df["patient_id"], test_size=0.20)
# train.head(3)

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    sgkf.split(
        df,
        y=df["expert_consensus"],
        groups=df["patient_id"]
    )
)

train = df.iloc[train_idx].reset_index(drop=True)
val = df.iloc[val_idx].reset_index(drop=True)
train.head(3)

In [ ]:
train = train[:train_len] 
val = val[:val_len]

In [ ]:
# p = 0
# u = df["patient_id"].unique()
# for i in u:
#     l = len(df[df["patient_id"] == i])
#     p += l
# p = p/len(u)
# p

In [ ]:
ax = sns.histplot(
    data=df,
    x="expert_consensus",
    hue="expert_consensus",
    palette=colors,
    alpha=0.3,
    element="bars",
    shrink=0.8,
    discrete=True
)

for patch in ax.patches:
    height = patch.get_height()

    if height > 0:
        ax.annotate(
            f"{int(height)}",
            (
                patch.get_x() + patch.get_width() / 2,
                height - 1500
            ),
            ha="center",
            va="bottom",
            xytext=(0, 3),
            textcoords="offset points"
        )

sns.move_legend(
    ax,
    "upper left",
    bbox_to_anchor=(1.02, 1),
    borderaxespad=0
)
plt.show()

In [ ]:
class ClassificationDataset(Dataset):                                           # Exactly 50 seconds of EEG data
    def __init__(self, df, spectrogram_dir, eeg_dir, target_cols=None, is_test=False, window_seconds=50, output_size=(128, 256)):
        self.df = df.reset_index(drop=True)
        self.spectrogram_dir = spectrogram_dir
        self.eeg_dir = eeg_dir
        self.target_cols = target_cols
        self.is_test = is_test
        self.window_seconds = window_seconds
        self.output_size = output_size

    def __len__(self):
        return len(self.df)

    # def __getitem__(self, idx):
    #     row = self.df.iloc[idx]

    #     spectrogram_id = row["spectrogram_id"]
    #     eeg_id = row["eeg_id"]

    #     sp_path = os.path.join(self.spectrogram_dir, f"{spectrogram_id}.parquet")
    #     eeg_path = os.path.join(self.eeg_dir, f"{eeg_id}.parquet")

    #     spectrogram_df = pd.read_parquet(sp_path)
    #     eeg_df = pd.read_parquet(eeg_path)

    #     spectrogram = spectrogram_df.to_numpy(dtype=np.float32)
    #     eeg = eeg_df.to_numpy(dtype=np.float32)

    #     spectrogram = torch.tensor(spectrogram, dtype=torch.float32)
    #     eeg = torch.tensor(eeg, dtype=torch.float32)

    #     X = {
    #         "spectrogram": spectrogram,
    #         "eeg": eeg
    #     }

    #     if self.is_test:
    #         return X

    #     y = torch.tensor(
    #         row[self.target_cols].values.astype(np.float32),
    #         dtype=torch.float32
    #     )

    #     return X, y

    # def _load_eeg(self, row):
    #     pass

    def _load_spectrogram(self, row):
        spectrogram_id = row["spectrogram_id"]
        sp_path = os.path.join(self.spectrogram_dir, f"{spectrogram_id}.parquet")
        raw = pd.read_parquet(sp_path)

        time = raw["time"].to_numpy(dtype=np.float32)
        values = raw.drop(columns="time").to_numpy(dtype=np.float32)

        if not self.is_test:
            start = float(row["spectrogram_label_offset_seconds"])
            end = start + self.window_seconds

            mask = (time >= start) & (time < end)
            values = values[mask]

        spec = values.T
        spec = np.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0)

        low, high = np.percentile(spec, [1, 99])
        spec = np.clip(spec, low, high)

        spec = (spec - spec.mean()) / (spec.std() + 1e-6)

        x = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        x = F.interpolate(x, size=self.output_size, mode="bilinear", align_corners=False)
        return x.squeeze(0)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        x = self._load_spectrogram(row)

        if self.is_test:
            return x

        votes = row[self.target_cols].values.astype(np.float32)

        y = votes / max(votes.sum(), 1e-8)
        y = torch.tensor(y, dtype=torch.float32)

        return x, y

In [ ]:
train_dataset = ClassificationDataset(train, spectogram_train_dir, eegs_train_dir, target_cols, is_test=False)
val_dataset = ClassificationDataset(val, spectogram_train_dir, eegs_train_dir, target_cols, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## Model

In [ ]:
model = timm.create_model(
    "resnet50",
    pretrained=False,
    in_chans=1,
    num_classes=6
).to(device)
opt = torch.optim.AdamW(model.parameters(),lr=1e-3, weight_decay=1e-4)

### Train

In [ ]:
loss_curve = []

for epoch in range(1, n_epochs+1):
    model.train()
    
    print(f"Epoch: {epoch}/{n_epochs}")
    running_loss = 0.0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        opt.zero_grad()
        logits = model(x)
        loss = kl_loss(logits, y)
        
        if torch.isfinite(loss):
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )
            
            opt.step()
            running_loss += loss.item() * x.size(0)
            
    loss_curve = np.append(loss_curve, running_loss)

plt.plot(loss_curve)
plt.show()

In [ ]:
torch.save(model, "/kaggle/working/model.pth")

### Vall

In [ ]:
result = {}
all_true = []
all_pred = []

model.eval()
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        
        logits = model(x)
        preds = torch.softmax(logits, dim=1)

        all_true.append(y.numpy())
        all_pred.append(preds.cpu().numpy())

y_true = np.concatenate(all_true, axis=0)
y_pred = np.concatenate(all_pred, axis=0)

true_classes = y_true.argmax(axis=1)
pred_classes = y_pred.argmax(axis=1)

score = metrics(true_classes, pred_classes)    
result = {
        "accuracy": score[0],
        "uar": score[1],
        "macro_f1": score[2],
    }
result

## Predict

In [ ]:
model_pth = "models/resnet50_prF"
model = torch.load(model_pth, map_location=device, weights_only=False)

In [ ]:
test = pd.read_csv("/kaggle/input/competitions/hms-harmful-brain-activity-classification/test.csv")
test

In [ ]:
test_dataset = ClassificationDataset(test, spectogram_test_dir, eegs_test_dir, target_cols, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
all_preds = []

with torch.no_grad():
    for x in test_loader:
        x = x.to(device)

        logits = model(x)
        preds = torch.softmax(logits, dim=1)

        all_preds.append(preds.cpu().numpy())
all_preds = np.concatenate(all_preds, axis=0)

submission = pd.DataFrame(
    all_preds,
    columns=target_cols
)
submission.insert(0, "eeg_id", test["eeg_id"].values)
submission

In [ ]:
submission.to_csv("/kaggle/working/submission.csv")

---